# RAG chunking: effect sizes and reranker transfer

Two analyses of the archived retrieval experiments:

1. Effect-size regression and minimum detectable effect (`scripts/20_effect_size.py`, CPU).
2. Transfer of the NQ-tuned reranker to TriviaQA (`scripts/21_reranker_transfer.py`, GPU).

Requires a GPU runtime, Drive access, the archived results, and the training and evaluation caches. If the fine-tuned weights are absent, the notebook automatically retrains the reranker before the transfer evaluation.


## Setup: Drive and project paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = os.environ.get('RAG_PROJECT_DIR')
if not PROJECT_DIR:
    raise RuntimeError('Set RAG_PROJECT_DIR to the project directory before running this notebook.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

## Install project dependencies


In [ ]:
!pip install -q -r requirements.txt

## Effect sizes and minimum detectable effect

Reads the Stage 6 archive and writes `artifacts/results/portfolio/effect_size_{report.md,coefficients.csv,tornado.png}`. Aggregate results cannot establish equivalence between methods.


In [ ]:
!python scripts/20_effect_size.py

## Fine-tuned reranker checkpoint

Training runs only when `bge_reranker_ft/final` contains no weights file.


In [ ]:
# Reuse existing weights; retrain from the cached Stage 8 groups only when absent.
import os, sys, pathlib, subprocess
ft = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'models' / 'bge_reranker_ft' / 'final'
def _weights(d):
    return list(d.glob('*.safetensors')) + list(d.glob('pytorch_model.bin'))
w = _weights(ft)
if w:
    print('FT reranker weights present:', [f.name for f in w])
else:
    print('FT reranker weights MISSING at', ft)
    print('Retraining scripts/17_train_reranker.py...')
    r = subprocess.run([sys.executable, 'scripts/17_train_reranker.py'])
    if r.returncode != 0:
        raise SystemExit('reranker training failed — see output above')
    w = _weights(ft)
    assert w, f'training finished but no weights file in {ft}'
    print('Retrained OK. weights now:', [f.name for f in w])

## Cross-dataset transfer

Evaluates five configurations and three arms on TriviaQA with a shared BGE top-20 pool. Completed configurations are checkpointed. Outputs use `artifacts/results/latest/stage8_transfer_*`.


In [ ]:
!python scripts/21_reranker_transfer.py

## Transfer results


In [ ]:
from IPython.display import Image, Markdown, display
import pathlib, os
latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
display(Markdown((latest / 'stage8_transfer_summary.md').read_text(encoding='utf-8')))
display(Image(str(latest / 'stage8_transfer_delta.png')))

## Outputs

- `artifacts/results/portfolio/effect_size_report.md`, coefficient CSV, and figure.
- `artifacts/results/latest/stage8_transfer_summary.md`, result CSV, and figure.

If the checkpoint was retrained, the transfer results use that new model, while the in-domain comparison uses the original Stage 8 archive. Differences may therefore include training variability as well as dataset effects.
